In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

tf.random.set_seed(42)
np.random.seed(42)


(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]


model_before = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax")
])

model_before.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining BEFORE regularization...\n")

history_before = model_before.fit(
    x_train,
    y_train,
    epochs=25,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

loss_before, acc_before = model_before.evaluate(x_test, y_test, verbose=0)


model_after = Sequential([
    Conv2D(
        32,
        (3, 3),
        activation="relu",
        kernel_regularizer=l2(0.0005),
        input_shape=(28, 28, 1)
    ),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.20),

    Conv2D(
        64,
        (3, 3),
        activation="relu",
        kernel_regularizer=l2(0.0005)
    ),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),

    Dense(
        128,
        activation="relu",
        kernel_regularizer=l2(0.0005)
    ),
    BatchNormalization(),
    Dropout(0.40),

    Dense(10, activation="softmax")
])

model_after.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

print("\nTraining AFTER regularization...\n")

history_after = model_after.fit(
    x_train,
    y_train,
    epochs=40,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

loss_after, acc_after = model_after.evaluate(x_test, y_test, verbose=0)


y_pred_before = np.argmax(model_before.predict(x_test), axis=1)
y_pred_after = np.argmax(model_after.predict(x_test), axis=1)


train_acc_before = history_before.history["accuracy"][-1]
val_acc_before = history_before.history["val_accuracy"][-1]
gap_before = train_acc_before - val_acc_before

best_val_acc_before = max(history_before.history["val_accuracy"])

train_acc_after = history_after.history["accuracy"][-1]
val_acc_after = history_after.history["val_accuracy"][-1]
gap_after = train_acc_after - val_acc_after

best_val_acc_after = max(history_after.history["val_accuracy"])


print("\n========== FINAL COMPARISON ==========")
print(f"{'Metric':<25}{'Before':<15}{'After':<15}")
print("-" * 55)
print(f"{'Test Accuracy':<25}{acc_before:<15.4f}{acc_after:<15.4f}")
print(f"{'Test Loss':<25}{loss_before:<15.4f}{loss_after:<15.4f}")
print(f"{'Final Train Accuracy':<25}{train_acc_before:<15.4f}{train_acc_after:<15.4f}")
print(f"{'Final Val Accuracy':<25}{val_acc_before:<15.4f}{val_acc_after:<15.4f}")
print(f"{'Best Val Accuracy':<25}{best_val_acc_before:<15.4f}{best_val_acc_after:<15.4f}")
print(f"{'Overfitting Gap':<25}{gap_before:<15.4f}{gap_after:<15.4f}")
print("======================================")


print("\nClassification Report BEFORE Regularization:\n")
print(classification_report(y_test, y_pred_before, target_names=class_names))

print("\nClassification Report AFTER Regularization:\n")
print(classification_report(y_test, y_pred_after, target_names=class_names))


plt.figure(figsize=(9, 5))
plt.plot(history_before.history["accuracy"], label="Train Before")
plt.plot(history_before.history["val_accuracy"], label="Val Before")
plt.plot(history_after.history["accuracy"], label="Train After")
plt.plot(history_after.history["val_accuracy"], label="Val After")
plt.title("Accuracy Comparison: Before vs After Regularization")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(9, 5))
plt.plot(history_before.history["loss"], label="Train Loss Before")
plt.plot(history_before.history["val_loss"], label="Val Loss Before")
plt.plot(history_after.history["loss"], label="Train Loss After")
plt.plot(history_after.history["val_loss"], label="Val Loss After")
plt.title("Loss Comparison: Before vs After Regularization")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


metrics = ["Test Acc", "Best Val Acc", "Gap"]
before_values = [acc_before, best_val_acc_before, gap_before]
after_values = [acc_after, best_val_acc_after, gap_after]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, before_values, width, label="Before")
plt.bar(x + width/2, after_values, width, label="After")
plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Final Metrics Comparison")
plt.legend()
plt.grid(axis="y")
plt.show()


cm_before = confusion_matrix(y_test, y_pred_before)
cm_after = confusion_matrix(y_test, y_pred_after)

disp_before = ConfusionMatrixDisplay(
    confusion_matrix=cm_before,
    display_labels=class_names
)

disp_before.plot(xticks_rotation=45)
plt.title("Confusion Matrix Before Regularization")
plt.show()

disp_after = ConfusionMatrixDisplay(
    confusion_matrix=cm_after,
    display_labels=class_names
)

disp_after.plot(xticks_rotation=45)
plt.title("Confusion Matrix After Regularization")
plt.show()
